**Le poids de la vérifiabilité dans le jugement des LLMs - Une étude épistémique sur l'évaluation de l'information**

In [ ]:
# ==============================================================================
# 1. INSTALLATION DES DÉPENDANCES
# (À décommenter si vous exécutez ce notebook pour la première fois)
# ==============================================================================
# !pip install -q transformers accelerate datasets statsmodels scipy pandas numpy

# ==============================================================================
# 2. IMPORTATION DES BIBLIOTHÈQUES
# ==============================================================================
# Manipulation de données et fichiers
import pandas as pd
import numpy as np
import json
import re
import os
import time
import random
import hashlib
import itertools
from datetime import datetime

# Importation du dataset LIAR2
from datasets import load_dataset

# Intelligence Artificielle (Hugging Face & PyTorch)
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

# Statistiques (Tests non paramétriques et Modèles Ordinaux)
from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Masquer les warnings de statsmodels pour garder le notebook propre
import warnings
warnings.filterwarnings("ignore")

print("Fait")

In [ ]:
# ==============================================================================
# ÉTAPE 2 : PRÉPARATION DU DATASET LIAR2
# ==============================================================================

# 1. Chargement de l'ensemble de test (contient environ 1200 à 2300 déclarations)
raw_data = load_dataset("chengxuphd/liar2", split="test") 

# 2. Constitution du bassin d'échantillonnage
# On fige l'aléatoire (random_state=42) pour la reproductibilité.
# Le modèle IA piochera dans ce bassin en fonction de son budget temps.
df = pd.DataFrame(raw_data).sample(n=2000, random_state=42)

# On ne garde que les métadonnées utiles à l'étude
df = df[["statement", "subject", "label"]].copy()

# 3. Binarisation de la vérité terrain (0 = Faux, 1 = Vrai)
# L'échelle originale LIAR2 va de 0 (Pants-fire) à 5 (True).
# Nous considérons comme "Vrai" (1) les labels 3 (Half-true), 4 (Mostly-true) et 5.
df["ground_truth"] = df["label"].apply(lambda x: 1 if x >= 3 else 0)

# 4. Sauvegarde pour l'étape d'inférence
nom_fichier = "verification_dataset.csv"
df.to_csv(nom_fichier, index=False, sep=';', encoding='utf-8-sig')

print(f"Fichier de base préparé et sauvegardé sous : {nom_fichier}")

df.shape
df.head(2)

> **Note :** Avant d'exécuter le code, vous pouvez pré-télécharger le modèle directement depuis votre terminal avec la commande `hf download deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` (ou `hf download deepseek-ai/DeepSeek-R1-Distill-Llama-8B` pour le plus grand modèle).

In [ ]:
# ==============================================================================
# ÉTAPE 3 : CHARGEMENT DU MODÈLE LLM (LOCAL)
# ==============================================================================

# 1. Configuration matérielle (Optimisation pour Apple Silicon)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"{device.upper()}")

# 2. Identification du modèle (Poids légers ~3 Go)
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"Chargement de {model_id} depuis le cache local")

# 3. Chargement du tokenizer et du modèle en précision réduite (float16)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device)

print("Fin")

In [ ]:
# ==============================================================================
# CELLULE : MOTEUR D'INFÉRENCE ET EXTRACTION JSON ROBUSTE
# /!\ AVERTISSEMENT : L'exécution complète prend ~60 min sur puce Apple Silicon.
# ==============================================================================

# Vérification de sécurité : le modèle doit être en mémoire (voir cellule précédente)
assert "model" in globals() and "tokenizer" in globals(), "Chargez d'abord le modèle !"

# ---------- 1. RÉGLAGES ----------
CHEMIN_CSV    = "verification_dataset.csv"
BUDGET_MIN    = 60                             # Durée de la collecte en minutes
SEED          = 42
SORTIE_BRUTE  = "liar2_brut.jsonl"             # Sauvegarde anti-crash (écrit à chaque phrase)
SORTIE_FINALE = "liar2_resultats_finaux.json"  # Fichier final pour les statistiques

PROMPT_VERSION = "v11-liar2-corrige"
GEN_PARAMS = {"temperature": 0.6, "top_p": 0.95, "max_new_tokens": 4096}
META_ENV = {
    "moteur": "transformers", "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "device": str(model.device), "dtype": str(model.dtype),
    "checkpoint": model.config._name_or_path,
}

# ---------- 2. CHARGEMENT SÉCURISÉ DU CSV ----------
print(f"Chargement des données depuis {CHEMIN_CSV}...")
data = pd.read_csv(CHEMIN_CSV, sep=";", encoding="utf-8-sig")
data = data.dropna(subset=["statement"]).reset_index(drop=True)

ITEMS, GT, SUBJ = [], {}, {}
for i, row in data.iterrows():
    iid = f"liar2_{i}"
    ITEMS.append((iid, str(row["statement"]), int(row["label"])))
    GT[iid] = int(row["ground_truth"])
    SUBJ[iid] = str(row["subject"])

print(f"{len(ITEMS)} phrases prêtes à être analysées.")

# Items de contrôle (Vérification de la vigilance basique)
CONTROLES = [
    ("ctrl_vrai_1",    "L'eau bout à 100 °C au niveau de la mer.", "vrai"),
    ("ctrl_vrai_2",    "Paris est la capitale de la France.", "vrai"),
    ("ctrl_faux_1",    "Le Soleil tourne autour de la Terre en 24 heures.", "faux"),
    ("ctrl_faux_2",    "Un kilogramme de plomb pèse moins qu'un kilogramme de plumes.", "faux"),
    ("ctrl_inverif_1", "Mon voisin a eu exactement trois pensées agréables hier à 15 h.", "inverifiable"),
    ("ctrl_inverif_2", "Il y a 4000 ans, un berger anonyme a appelé sa chèvre Ada.", "inverifiable"),
]

# ---------- 3. DÉFINITION DU PROMPT ET DES GABARITS ----------
DIMS = {
    "verifiabilite": '"verifiabilite" : À quel point est-il facile (rapide, peu coûteux, sans expertise) de vérifier empiriquement cette affirmation ? (1 = totalement impossible, 4 = moyennement facile, 7 = vérification immédiate).',
    "plausibilite": '"plausibilite" : À quel point cette affirmation est-elle vraisemblable selon tes connaissances du monde ? (1 = totalement invraisemblable, 4 = ni vraisemblable ni invraisemblable, 7 = tout à fait vraisemblable).',
    "croyance": '"croyance" : À quel point estimes-tu que cette affirmation est vraie ? (1 = certitude absolue que c\'est faux, 4 = aucune idée si c\'est vrai ou faux, 7 = certitude absolue que c\'est vrai).',
}
PERMUTATIONS = list(itertools.permutations(DIMS.keys()))

def generer_prompt(texte, cles):
    questions = "\n".join(f"{i+1}. {DIMS[k]}" for i, k in enumerate(cles))
    schema = ",\n".join(f'  "{k}": <entier de 1 à 7>' for k in cles)
    return (f"Lis l'affirmation suivante : « {texte} »\n\n"
            f"Évalue ces trois dimensions sur une échelle entière de 1 à 7 :\n\n{questions}\n\n"
            "Ta réponse finale DOIT ÊTRE STRICTEMENT ce dictionnaire JSON exact, "
            f"sans aucun texte supplémentaire :\n{{\n{schema}\n}}")

HASH_GABARITS = {"|".join(o): hashlib.sha256(generer_prompt("", o).encode()).hexdigest()[:16] for o in PERMUTATIONS}
CONFIG_ID = hashlib.sha256(json.dumps([PROMPT_VERSION, GEN_PARAMS, sorted(HASH_GABARITS.items())], sort_keys=True).encode()).hexdigest()[:10]

# ---------- 4. EXTRACTEUR JSON ROBUSTE ----------
def extraire_json_robuste(texte_final, cles):
    m = re.search(r"\{[^{}]*\}", texte_final, re.DOTALL)
    if not m: return None, "regex_non_trouvee"
    try: d = json.loads(m.group())
    except json.JSONDecodeError: return None, "json_decode_error"
    
    if not isinstance(d, dict): return None, "pas_un_dict"
    if set(d) != set(cles): return None, f"cles_inattendues_{sorted(set(d) ^ set(cles))}"
    
    for k in cles:
        v = d[k]
        if type(v) is not int: return None, f"type_errone_{k}_{type(v).__name__}"
        if not 1 <= v <= 7: return None, f"hors_bornes_{k}_{v}"
    return {k: d[k] for k in cles}, "ok"

# ---------- 5. MOTEUR D'ANNOTATION (IA) ----------
def annoter_item(texte, item_id, j_ordre, seed_global=SEED):
    ordre = PERMUTATIONS[j_ordre]
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": generer_prompt(texte, ordre)}], tokenize=False, add_generation_prompt=True)
    prefill = prompt.rstrip().endswith("<think>")
    
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    L = inputs.input_ids.shape[1]

    seed_locale = int(hashlib.sha256(f"{seed_global}|{item_id}|{j_ordre}".encode()).hexdigest(), 16) % (2**31)
    torch.manual_seed(seed_locale)
    
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, do_sample=True, pad_token_id=tokenizer.eos_token_id, **GEN_PARAMS)
    duree = time.perf_counter() - t0

    ids = out[0][L:]
    pos_eos = (ids == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    eos_vu = len(pos_eos) > 0
    n_tok = int(pos_eos[0]) + 1 if eos_vu else len(ids)
    txt = tokenizer.decode(ids[:n_tok], skip_special_tokens=False).replace(tokenizer.eos_token, "").strip()

    ferme = "</think>" in txt
    res, cause = (extraire_json_robuste(txt.rsplit("</think>", 1)[1], ordre) if ferme else (None, "sans_think_ferme"))
    statut = "tronque" if not eos_vu else ("sans_think_ferme" if not ferme else ("ok" if res else cause))
    trace = txt.split("</think>")[0].replace("<think>", "").strip() if ferme else ""
    
    vals = {k: None for k in DIMS}
    vals.update(res or {})

    return {
        "item_id": item_id, "condition": "conjoint", "ordre": "|".join(ordre), "ordre_idx": j_ordre,
        "rep": 0, "seed_global": seed_global, "seed_locale": seed_locale,
        "prompt_version": PROMPT_VERSION, "config_id": CONFIG_ID,
        "hash_gabarit": HASH_GABARITS["|".join(ordre)],
        "prefill_think": prefill, "n_tokens_generes": n_tok, "duree_s": round(duree, 1),
        "statut": statut, **vals, "trace_pensee": trace, "sortie_brute": txt,
        **GEN_PARAMS, **META_ENV,
    }

# ---------- 6. FONCTION DE COLLECTE ET AFFICHAGE ----------
try: from IPython.display import clear_output
except ImportError: clear_output = None

def _afficher(n, elapsed, budget_s, statuts, durees):
    pct = min(elapsed / budget_s, 1); plein = int(30 * pct)
    moy = sum(durees) / len(durees)
    if clear_output: clear_output(wait=True)
    print(f"[{'█'*plein}{'░'*(30-plein)}] {pct:4.0%} | {n} items analysés | écoulé {elapsed/60:4.1f} min | "
          f"restant ~{max(budget_s-elapsed,0)/60:4.1f} min | {moy:3.0f} s/item | "
          f"Réussis (ok) : {statuts.count('ok')}/{n} | Échoués (tronqués) : {statuts.count('tronque')}", flush=True)

def collecte(budget_s=BUDGET_MIN * 60, chemin=SORTIE_BRUTE):
    faits = set()
    if os.path.exists(chemin):
        with open(chemin, encoding="utf-8") as f:
            for l in f:
                try: d = json.loads(l); faits.add((d["item_id"], d.get("config_id")))
                except: pass

    echantillon = list(ITEMS)
    random.Random(SEED).shuffle(echantillon) 
    file_att = ([(i, t, None, True, a) for i, t, a in CONTROLES] + [(i, t, lab, False, None) for i, t, lab in echantillon])

    t0, n, statuts, durees = time.time(), 0, [], []
    try:
        with open(chemin, "a", encoding="utf-8") as f:
            for k, (iid, texte, lab, est_ctrl, attendu) in enumerate(file_att):
                if (iid, CONFIG_ID) in faits: continue
                moy = sum(durees) / len(durees) if durees else 0
                if time.time() - t0 + moy > budget_s: break
                
                t_item = time.time()
                ligne = annoter_item(texte, iid, j_ordre=k % 6)
                
                ligne.update({
                    "texte": texte, "label_liar2": lab, "label_liar2_texte": str(lab) if lab is not None else None,
                    "ground_truth": GT.get(iid), "subject": SUBJ.get(iid),
                    "est_controle": est_ctrl, "attendu_controle": attendu
                })
                
                f.write(json.dumps(ligne, ensure_ascii=False) + "\n"); f.flush()
                n += 1; durees.append(time.time() - t_item); statuts.append(ligne["statut"])
                _afficher(n, time.time() - t0, budget_s, statuts, durees)
    except KeyboardInterrupt:
        print("\n⏹ Interrompu manuellement : les données traitées sont sauvegardées.")
    print(f"\n Terminé : {n} items traités en {(time.time()-t0)/60:.1f} minutes.")

# ---------- 7. EXPORT DU FICHIER PROPRE ET BILAN STATISTIQUE ----------
def exporter_json_et_bilan(brut=SORTIE_BRUTE, sortie=SORTIE_FINALE):
    uniques = {}
    with open(brut, encoding="utf-8") as f:
        for l in f:
            try: d = json.loads(l)
            except json.JSONDecodeError: continue
            if d.get("config_id") == CONFIG_ID:
                uniques[(d["item_id"], d["condition"], d["ordre"], d["rep"])] = d
                
    champs = ["item_id", "est_controle", "attendu_controle", "texte", "label_liar2", "label_liar2_texte",
              "ground_truth", "subject", "ordre", "statut", "verifiabilite", "plausibilite", "croyance", "n_tokens_generes", "duree_s"]
    resultats = [{c: d.get(c) for c in champs} for d in uniques.values()]
    comptes = pd.Series([r["statut"] for r in resultats]).value_counts().to_dict()
    
    with open(sortie, "w", encoding="utf-8") as f:
        json.dump({"meta": {"n_lignes": len(resultats), "statuts": comptes}, "resultats": resultats}, f, ensure_ascii=False, indent=2)
    
    df = pd.DataFrame(resultats)
    print(f"\n JSON propre généré : {sortie} ({len(resultats)} lignes conservées)")
    print("\n=== Qualité de l'extraction JSON ===")
    print(df["statut"].value_counts().to_string())
    
    ok = df[(df.statut == "ok") & (df.est_controle != True)].copy()
    print(f"\n=== Analyse sur les données LIAR2 (N={len(ok)}) ===")
    if len(ok) > 0 and ok["ground_truth"].notna().any():
        print("Note de croyance moyenne selon la Vraie Véracité (0 = Faux, 1 = Vrai) :")
        print(ok.groupby("ground_truth")["croyance"].agg(["mean", "count"]).round(2).to_string())

# ==============================================================================
# 8. LANCEMENT (Commenté pour éviter un écrasement accidentel sur GitHub)
# ==============================================================================
# collecte()
# exporter_json_et_bilan()

In [ ]:
# ==============================================================================
# CELLULE : ANALYSE STATISTIQUE ET MODÉLISATION ORDINALE
# ==============================================================================
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings

# 1. Chargement et préparation des données
df = pd.DataFrame(json.load(open("liar2_resultats_finaux.json", encoding="utf-8"))["resultats"])
ok = df[(df.statut == "ok") & (df.est_controle != True)].copy()
V = ["verifiabilite", "plausibilite", "croyance"]
ok[V] = ok[V].astype(int)
rng = np.random.default_rng(42)

print(f"=== ANALYSE STATISTIQUE SUR N = {len(ok)} ITEMS LIAR2 ===")

# 2. Descriptif des réponses (Le repli sur l'incertitude)
print("\n--- 1. Répartition des notes (1 à 7) ---")
print(ok[V].apply(lambda s: s.value_counts(normalize=True)).round(2).fillna(0).sort_index())
print("\nPart de triplets neutres (4,4,4) :", (ok[V] == 4).all(axis=1).mean().round(2))

# 3. Discrimination Vrai/Faux
sub = ok[ok.label_liar2 != 3] # Exclusion des 'Half-True' (ambigus)
a = sub.loc[sub.ground_truth == 1, "croyance"].to_numpy()
b = sub.loc[sub.ground_truth == 0, "croyance"].to_numpy()
auc = lambda x, y: mannwhitneyu(x, y).statistic / (len(x) * len(y))
bs = [auc(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(2000)]

print(f"\n--- 2. Discrimination Vrai/Faux (Exclusion Half-True) ---")
print(f"AUC = {auc(a, b):.2f} | IC95 {np.percentile(bs, [2.5, 97.5]).round(2)}")

# 4. Corrélations de rang (Spearman)
def ic(f, x, y, B=2000):
    r = []
    for _ in range(B):
        i = rng.integers(0, len(x), len(x)); v = f(x[i], y[i])[0]
        if not np.isnan(v): r.append(v)
    return np.percentile(r, [2.5, 97.5]).round(2)

print("\n--- 3. Corrélations brutes de Spearman ---")
print(ok[V].corr(method="spearman").round(2))
x, y = ok.verifiabilite.to_numpy(), ok.croyance.to_numpy()
print(f"\nSpearman (Vérifiabilité × Croyance) : {spearmanr(x, y)[0]:.2f} | IC95 {ic(spearmanr, x, y)}")

# 5. Modélisation Ordinale (La chute de l'effet de vérifiabilité)
print("\n--- 4. Modèles Ordinaux (Prédiction de la Croyance) ---")
try:
    m0 = OrderedModel(ok["croyance"], ok[["verifiabilite"]], distr="logit").fit(method="bfgs", disp=False)
    print("\nModèle A (Vérifiabilité seule) :")
    print(f"Coef: {m0.params['verifiabilite']:.3f} | IC95: {m0.conf_int().loc['verifiabilite'].round(3).tolist()}")

    m1 = OrderedModel(ok["croyance"], ok[["verifiabilite", "plausibilite"]], distr="logit").fit(method="bfgs", disp=False)
    print("\nModèle B (Vérifiabilité ajustée sur la Plausibilité) :")
    print(m1.summary().tables[1])
except Exception as e:
    print(f"Le modèle ordinal n'a pas convergé : {e}")

# 6. Écart au point neutre (Analyse exploratoire)
ok["ecart"] = (ok.croyance != 4).astype(int)
print("\n--- 5. Régression Logistique : S'écarter du 4 (Incertitude) ---")
print(smf.logit("ecart ~ verifiabilite + plausibilite", ok).fit(disp=False).summary().tables[1])

**FIN**